
# Endogenous Reachability Collapse (ERC)
## Full model notebook aligned with the manuscript

This notebook contains the full Python model used to generate the ERC results discussed in the manuscript.

### Falseador note
This is a **phenomenological proof-of-principle model**. It supports:
- contraction of the recoverable set
- dynamic loss of recoverability
- **path-dependent / hysteresis-like** behavior
- state-dependent threshold shift

It does **not** claim:
- formal topological elimination of trajectories
- mathematical proof of irreversibility
- universal linearity of the threshold shift beyond the explored range
- direct mapping to a specific biological system


In [ ]:

# Imports
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# Optional: make plots a bit cleaner
plt.rcParams["figure.dpi"] = 120


## 0. Global parameters

In [ ]:

# Core model parameters
omega = 2.0
a0 = 1.0
alpha = 1.2
b = 2.0

# Recovery criterion
r_recovery = 0.15

# Fixed-c simulation settings
t_max = 40.0
n_eval = 2000

print("omega =", omega)
print("a0 =", a0)
print("alpha =", alpha)
print("b =", b)
print("r_recovery =", r_recovery)


## 1. Core ERC model

In [ ]:

def a_of_c(c, a0=a0, alpha=alpha):
    return max(0.05, a0 - alpha * c)

def erc_rhs(t, state, c, omega=omega, a0=a0, alpha=alpha, b=b):
    x, y = state
    r2 = x*x + y*y
    a = a_of_c(c, a0=a0, alpha=alpha)
    g = -((r2 - a*a) * (r2 - b*b))
    dx = g * x - omega * y
    dy = g * y + omega * x
    return [dx, dy]

def simulate_trajectory(x0, y0, c, t_max=t_max, n_eval=n_eval, rtol=1e-7, atol=1e-9):
    t_eval = np.linspace(0, t_max, n_eval)
    sol = solve_ivp(
        erc_rhs,
        [0, t_max],
        [x0, y0],
        args=(c,),
        t_eval=t_eval,
        rtol=rtol,
        atol=atol,
    )
    x = sol.y[0]
    y = sol.y[1]
    r = np.sqrt(x**2 + y**2)
    return sol.t, x, y, r

def recovered_to_base(r, r_recovery=r_recovery):
    return np.any(r < r_recovery)


In [ ]:

# Quick sanity check
c_test = 0.0
x0_test, y0_test = 0.5, 0.0

t, x, y, r = simulate_trajectory(x0_test, y0_test, c_test)

print("Recovered:", recovered_to_base(r))

plt.figure(figsize=(6, 4))
plt.plot(t, r)
plt.axhline(r_recovery, linestyle="--")
plt.xlabel("t")
plt.ylabel("r(t)")
plt.title(f"Test trajectory, c={c_test}")
plt.show()


## 2. Basin contraction analysis

In [ ]:

# Refined basin scan settings
t_max_ref = 25.0
n_eval_ref = 1000

x_vals_ref = np.linspace(-2.5, 2.5, 81)
y_vals_ref = np.linspace(-2.5, 2.5, 81)
c_values_ref = np.linspace(0.0, 0.7, 6)

def simulate_trajectory_ref(x0, y0, c):
    t_eval = np.linspace(0, t_max_ref, n_eval_ref)
    sol = solve_ivp(
        erc_rhs,
        [0, t_max_ref],
        [x0, y0],
        args=(c,),
        t_eval=t_eval,
        rtol=1e-6,
        atol=1e-8,
        method="RK45",
    )
    x = sol.y[0]
    y = sol.y[1]
    r = np.sqrt(x**2 + y**2)
    return r

def recovered_ref(r, r_recovery=0.15):
    return (np.any(r < r_recovery) and (r[-1] < r_recovery))

def basin_map_ref(c, x_vals, y_vals):
    basin = np.zeros((len(y_vals), len(x_vals)), dtype=int)
    for i, y0 in enumerate(y_vals):
        if i % 10 == 0:
            print(f"  c={c:.2f} | row {i+1}/{len(y_vals)}")
        for j, x0 in enumerate(x_vals):
            r = simulate_trajectory_ref(x0, y0, c)
            basin[i, j] = 1 if recovered_ref(r) else 0
    return basin


Run the next cell to compute basin maps. It is one of the heaviest parts of the notebook.

In [ ]:

basin_maps_ref = {}
recoverable_fraction_ref = []

for c in c_values_ref:
    print(f"\nComputing REFINED basin map for c = {c:.2f}")
    basin = basin_map_ref(c, x_vals_ref, y_vals_ref)
    basin_maps_ref[c] = basin
    recoverable_fraction_ref.append(basin.mean())

print("\nDone.")


In [ ]:

fig = plt.figure(figsize=(12, 6))

selected_cs = [c_values_ref[0], c_values_ref[2], c_values_ref[-1]]

for i, c in enumerate(selected_cs):
    ax = plt.subplot(2, 3, i + 1)
    basin = basin_maps_ref[c]
    ax.imshow(
        basin,
        extent=[x_vals_ref[0], x_vals_ref[-1], y_vals_ref[0], y_vals_ref[-1]],
        origin="lower",
        aspect="equal",
    )
    ax.set_title(f"c = {c:.2f}")
    ax.set_xlabel("x0")
    ax.set_ylabel("y0")

ax_curve = plt.subplot(2, 1, 2)
ax_curve.plot(c_values_ref, recoverable_fraction_ref, marker="o")
ax_curve.set_xlabel("Internal constraint c")
ax_curve.set_ylabel("Recoverable fraction")
ax_curve.set_title("Contraction of recoverable set")
ax_curve.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("figure_1_basin_contraction.png", dpi=300, bbox_inches="tight")
plt.show()


## 3. Ring-based recovery-time analysis

In [ ]:

def recovery_time(r, t, r_recovery=0.15):
    idx = np.where(r < r_recovery)[0]
    if len(idx) == 0:
        return np.nan
    return t[idx[0]]

def simulate_trajectory_ref_full(x0, y0, c):
    t_eval = np.linspace(0, t_max_ref, n_eval_ref)
    sol = solve_ivp(
        erc_rhs,
        [0, t_max_ref],
        [x0, y0],
        args=(c,),
        t_eval=t_eval,
        rtol=1e-6,
        atol=1e-8,
        method="RK45",
    )
    x = sol.y[0]
    y = sol.y[1]
    r = np.sqrt(x**2 + y**2)
    return sol.t, r

r0_min = 0.8
r0_max = 1.2

mean_recovery_times_ring = []
median_recovery_times_ring = []
recoverable_fraction_ring = []

for c in c_values_ref:
    times_c = []
    total_ring = 0
    recovered_ring = 0

    print(f"\nComputing ring-based recovery times for c = {c:.2f}")

    for i, y0 in enumerate(y_vals_ref):
        if i % 10 == 0:
            print(f"  c={c:.2f} | row {i+1}/{len(y_vals_ref)}")
        for j, x0 in enumerate(x_vals_ref):
            r0 = np.sqrt(x0**2 + y0**2)
            if r0_min < r0 < r0_max:
                total_ring += 1
                t, r = simulate_trajectory_ref_full(x0, y0, c)
                tau = recovery_time(r, t, r_recovery=0.15)

                if not np.isnan(tau) and (r[-1] < 0.15):
                    recovered_ring += 1
                    times_c.append(tau)

    recoverable_fraction_ring.append(recovered_ring / total_ring if total_ring > 0 else np.nan)

    if len(times_c) == 0:
        mean_recovery_times_ring.append(np.nan)
        median_recovery_times_ring.append(np.nan)
    else:
        mean_recovery_times_ring.append(np.mean(times_c))
        median_recovery_times_ring.append(np.median(times_c))


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

axes[0].plot(c_values_ref, mean_recovery_times_ring, marker="o", label="Mean recovery time (ring)")
axes[0].plot(c_values_ref, median_recovery_times_ring, marker="s", label="Median recovery time (ring)")
axes[0].set_xlabel("Internal constraint c")
axes[0].set_ylabel("Recovery time")
axes[0].set_title("Recovery time vs internal constraint (fixed initial ring)")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(c_values_ref, recoverable_fraction_ring, marker="o")
axes[1].set_xlabel("Internal constraint c")
axes[1].set_ylabel("Recoverable fraction in ring")
axes[1].set_title("Recoverable fraction vs internal constraint (fixed initial ring)")
axes[1].grid(True, alpha=0.3)

plt.savefig("figure_2_ring_analysis.png", dpi=300, bbox_inches="tight")
plt.show()


## 4. Dynamic ERC trajectories

In [ ]:

eps_dyn = 0.08
t_max_dyn = 60.0
n_eval_dyn = 2000

def erc_rhs_dynamic(t, state, omega=omega, a0=a0, alpha=alpha, b=b, eps=eps_dyn):
    x, y, c = state
    r2 = x*x + y*y
    a = max(0.05, a0 - alpha * c)
    g = -((r2 - a*a) * (r2 - b*b))
    dx = g * x - omega * y
    dy = g * y + omega * x
    dc = eps * ((r2 / (1.0 + r2)) - c)
    return [dx, dy, dc]

def simulate_dynamic_erc(x0, y0, c0=0.0, t_max=t_max_dyn, n_eval=n_eval_dyn):
    t_eval = np.linspace(0, t_max, n_eval)
    sol = solve_ivp(
        erc_rhs_dynamic,
        [0, t_max],
        [x0, y0, c0],
        t_eval=t_eval,
        rtol=1e-6,
        atol=1e-8,
        method="RK45",
    )
    x = sol.y[0]
    y = sol.y[1]
    c = sol.y[2]
    r = np.sqrt(x**2 + y**2)
    a = np.maximum(0.05, a0 - alpha * c)
    margin = a - r
    return sol.t, x, y, r, c, a, margin


In [ ]:

initial_conditions_dyn = [
    (0.4, 0.0, "small initial radius"),
    (0.8, 0.0, "intermediate initial radius"),
    (1.1, 0.0, "larger initial radius"),
]

fig, axes = plt.subplots(len(initial_conditions_dyn), 4, figsize=(16, 4 * len(initial_conditions_dyn)), constrained_layout=True)

if len(initial_conditions_dyn) == 1:
    axes = np.array([axes])

for row, (x0, y0, label) in enumerate(initial_conditions_dyn):
    t, x, y, r, c, a, margin = simulate_dynamic_erc(x0, y0, c0=0.0)

    ax = axes[row, 0]
    ax.plot(t, r, label="r(t)")
    ax.plot(t, a, linestyle="--", label="a(c(t))")
    ax.set_title(f"{label}\nRadius vs recoverability boundary")
    ax.set_xlabel("t")
    ax.set_ylabel("radius")
    ax.legend()

    ax = axes[row, 1]
    ax.plot(t, c)
    ax.set_title("Internal constraint c(t)")
    ax.set_xlabel("t")
    ax.set_ylabel("c")

    ax = axes[row, 2]
    ax.plot(t, margin)
    ax.axhline(0, linestyle="--")
    ax.set_title("Dynamic safety margin: a(c(t)) - r(t)")
    ax.set_xlabel("t")
    ax.set_ylabel("margin")

    ax = axes[row, 3]
    ax.plot(x, y)
    ax.scatter([x0], [y0], marker="o", s=30, label="start")
    ax.scatter([x[-1]], [y[-1]], marker="x", s=40, label="end")
    ax.set_title("Trajectory in phase plane")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_aspect("equal")
    ax.legend()

plt.savefig("figure_3_dynamic_trajectories.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:

def classify_dynamic_trajectory(r, margin, r_recovery=0.15):
    recovered = (np.any(r < r_recovery) and (r[-1] < r_recovery))
    margin_crossed = np.any(margin < 0)
    return recovered, margin_crossed

for x0, y0, label in initial_conditions_dyn:
    t, x, y, r, c, a, margin = simulate_dynamic_erc(x0, y0, c0=0.0)
    recovered, margin_crossed = classify_dynamic_trajectory(r, margin)

    print(f"{label}:")
    print(f"  recovered = {recovered}")
    print(f"  margin crossed below zero = {margin_crossed}")
    print(f"  final c = {c[-1]:.4f}")
    print(f"  final r = {r[-1]:.4f}")
    print(f"  min margin = {margin.min():.4f}")
    print()


## 5. Path-dependent / hysteresis-like behavior

In [ ]:

def erc_rhs_path_dependence(t, state, omega=omega, a0=a0, alpha=alpha, b=b, eps=eps_dyn, phase_switch=30.0):
    x, y, c = state
    r2 = x*x + y*y
    a = max(0.05, a0 - alpha * c)
    g = -((r2 - a*a) * (r2 - b*b))
    dx = g * x - omega * y
    dy = g * y + omega * x

    if t < phase_switch:
        dc = eps * ((r2 / (1.0 + r2)) - c)
    else:
        dc = -0.02 * c

    return [dx, dy, dc]

def simulate_path_dependence(x0, y0, c0=0.0, t_max=80.0, n_eval=2500):
    t_eval = np.linspace(0, t_max, n_eval)
    sol = solve_ivp(
        erc_rhs_path_dependence,
        [0, t_max],
        [x0, y0, c0],
        t_eval=t_eval,
        rtol=1e-6,
        atol=1e-8,
    )
    x = sol.y[0]
    y = sol.y[1]
    c = sol.y[2]
    r = np.sqrt(x**2 + y**2)
    a = np.maximum(0.05, a0 - alpha * c)
    margin = a - r
    return sol.t, x, y, r, c, a, margin


In [ ]:

t, x, y, r, c, a, margin = simulate_path_dependence(1.2, 0.0)

fig, axes = plt.subplots(4, 1, figsize=(8, 10), constrained_layout=True)

axes[0].plot(t, r, label="r(t)")
axes[0].plot(t, a, linestyle="--", label="a(c(t))")
axes[0].axvline(30, linestyle=":", color="k", label="phase switch")
axes[0].set_title("Radius vs recoverability boundary")
axes[0].legend()

axes[1].plot(t, c)
axes[1].axvline(30, linestyle=":")
axes[1].set_title("Internal constraint c(t)")

axes[2].plot(t, margin)
axes[2].axhline(0, linestyle="--")
axes[2].axvline(30, linestyle=":")
axes[2].set_title("Dynamic safety margin")

axes[3].plot(x, y)
axes[3].scatter([x[0]], [y[0]], label="start")
axes[3].scatter([x[-1]], [y[-1]], label="end")
axes[3].set_aspect("equal")
axes[3].set_title("Trajectory in phase space")
axes[3].legend()

plt.savefig("figure_4_path_dependence.png", dpi=300, bbox_inches="tight")
plt.show()


## 6. State-dependent recoverability threshold

In [ ]:

r0_values = np.linspace(0.2, 1.5, 40)
c0_values = [0.0, 0.1, 0.2]

def initial_condition_from_radius(r0):
    return r0, 0.0

def analyze_radius(r0, c0):
    x0, y0 = initial_condition_from_radius(r0)
    t, x, y, r, c, a, margin = simulate_dynamic_erc(x0, y0, c0=c0)
    recovered = (np.any(r < 0.15) and (r[-1] < 0.15))
    min_margin = np.min(margin)
    return recovered, min_margin

results = {}

for c0 in c0_values:
    print(f"\nScanning for c0 = {c0}")
    recovered_list = []
    margin_list = []

    for r0 in r0_values:
        recovered, min_margin = analyze_radius(r0, c0)
        recovered_list.append(recovered)
        margin_list.append(min_margin)

    results[c0] = {
        "recovered": np.array(recovered_list),
        "min_margin": np.array(margin_list),
    }


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

for c0 in c0_values:
    recovered = results[c0]["recovered"]
    axes[0].plot(r0_values, recovered.astype(int), marker="o", label=f"c0={c0}")
axes[0].set_xlabel("Initial radius r0")
axes[0].set_ylabel("Recovered (1=yes, 0=no)")
axes[0].set_title("Recoverability vs initial radius")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

for c0 in c0_values:
    margin = results[c0]["min_margin"]
    axes[1].plot(r0_values, margin, marker="o", label=f"c0={c0}")
axes[1].axhline(0, linestyle="--")
axes[1].set_xlabel("Initial radius r0")
axes[1].set_ylabel("Minimum dynamic margin")
axes[1].set_title("Critical threshold vs initial radius")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.savefig("figure_5_threshold_scan.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:

critical_points = []

for c0 in c0_values:
    recovered = results[c0]["recovered"]
    idx = np.where(recovered == False)[0]
    rcrit = np.nan if len(idx) == 0 else r0_values[idx[0]]
    critical_points.append((c0, rcrit))

c0_crit = np.array([pt[0] for pt in critical_points])
rcrit = np.array([pt[1] for pt in critical_points])

mask = ~np.isnan(rcrit)
coef = np.polyfit(c0_crit[mask], rcrit[mask], 1)
slope, intercept = coef

print("c0:", c0_crit)
print("rcrit:", rcrit)
print(f"Approximate fit in explored range: r0* ≈ {intercept:.4f} {slope:+.4f}·c0")

c0_fit = np.linspace(c0_crit.min(), c0_crit.max(), 100)
rcrit_fit = slope * c0_fit + intercept

plt.figure(figsize=(6, 4))
plt.plot(c0_crit, rcrit, "o", label="Critical radius (measured)")
plt.plot(c0_fit, rcrit_fit, "-", label="Linear fit")

for x, y in zip(c0_crit, rcrit):
    plt.annotate(f"{y:.2f}", (x, y), textcoords="offset points", xytext=(5, 5))

plt.xlabel("Initial internal constraint c0")
plt.ylabel("Critical initial radius r0*")
plt.title("Shift of recoverability threshold with initial constraint")
plt.grid(True, alpha=0.3)
plt.legend()
plt.savefig("figure_6_threshold_shift_fit.png", dpi=300, bbox_inches="tight")
plt.show()



## Final falseador note

This notebook is aligned with the manuscript and contains the full code needed to support the published figures.

### Recommended manuscript language
- Use **path-dependent** or **hysteresis-like** behavior for Figure 4.
- Use **approximately linear in the explored range** for Figure 6.
- Present the ring analysis as a **controlled anti-bias analysis**, not as a universal property of the full state space.
